In [ ]:
!pip install -U pinecone sentence-transformers pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 742.7/742.7 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.4/512.4 kB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 118.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.9/280.9 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 7.4 MB/s eta 0:00:00
  Attempting uninstall: packaging
    Found existing installation: packaging 26.0
    Uninstalling packaging-26.0:
      Successfully uninstalled packaging-26.0
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 5.2.3
    Uninstalling sentence-transformers-5.2.3:
      Successfully uninstalled sentence-transformers-

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


## declare standard dense pincone index

In [ ]:
from pinecone import Pinecone, ServerlessSpec

PINECONE_API_KEY = "######"
pc = Pinecone(api_key=PINECONE_API_KEY)

index_name = "medical-sport-assistant"
dimension_size = 1024
metric_type = "cosine"

if index_name not in pc.list_indexes().names():
    print(f"Creating new index: '{index_name}'...")
    pc.create_index(
        name=index_name,
        dimension=dimension_size,
        metric=metric_type,
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )
    print("Index created successfully! It is now a standard Dense index.")
else:
    print(f"Index '{index_name}' already exists.")
    print("Please delete the old one in the Pinecone Dashboard first, then run this again.")

index = pc.Index(index_name)
print("\nCurrent Index Stats:")
print(index.describe_index_stats())

Creating new index: 'medical-sport-assistant'...
Index created successfully! It is now a standard Dense index.

Current Index Stats:
{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '151',
                                    'content-type': 'application/json',
                                    'date': 'Thu, 12 Mar 2026 18:39:43 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '41',
                                    'x-pinecone-request-latency-ms': '40',
                                    'x-pinecone-response-duration-ms': '42'}},
 'dimension': 1024,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'storageFullness': 0.0,
 'total_vector_count': 0,
 'vector_type': 'dense'}


# food and nutrition data

In [ ]:
import shutil
import os

file_path="/content/drive/MyDrive/SPORT_METADATA/Structure data/Food and nutrition/egypt_foods_metadata.csv"
dist_path="/content/food_data/"

os.makedirs("/content/food_data/",exist_ok=True)

shutil.copy2(file_path,dist_path)

'/content/food_data/egypt_foods_metadata.csv'

In [ ]:
import pandas as pd

df=pd.read_csv("/content/food_data/egypt_foods_metadata.csv")

print(df.columns)

Index(['search_query', 'fdc_id', 'food_name', 'food_category', 'data_type',
       'calories', 'protein_g', 'carbs_g', 'fat_g', 'fiber_g', 'sugar_g',
       'saturated_fat_g', 'cholesterol_mg', 'sodium_mg', 'iron_mg',
       'calcium_mg', 'potassium_mg', 'magnesium_mg', 'vitamin_c_mg',
       'vitamin_b12_ug', 'text_chunk'],
      dtype='str')


In [ ]:
df.iloc[0]

search_query                                     Chicken breast, raw
fdc_id                                                       2646170
food_name                   Chicken, breast, boneless, skinless, raw
food_category                                       Poultry Products
data_type                                                 Foundation
calories                                                         0.0
protein_g                                                       22.5
carbs_g                                                          0.0
fat_g                                                           1.93
fiber_g                                                          0.0
sugar_g                                                          0.0
saturated_fat_g                                                0.349
cholesterol_mg                                                  72.7
sodium_mg                                                       65.8
iron_mg                           

In [ ]:
print("the number of null value that may cause chrash pinecone",int(df.isna().sum().sum()))

the number of null value that may cause chrash pinecone 0


In [ ]:
import pandas as pd
from pinecone import Pinecone, ServerlessSpec
from sentence_transformers import SentenceTransformer
import torch

PINECONE_API_KEY = "######"
pc = Pinecone(api_key=PINECONE_API_KEY)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Loading model on: {device}")

model = SentenceTransformer('BAAI/bge-large-en-v1.5', device=device)
dimension_size = 1024
index_name = "medical-sport-assistant"

if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=dimension_size,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

index = pc.Index(index_name)

df = pd.read_csv("/content/food_data/egypt_foods_metadata.csv")

batch_size = 100
namespace_name = "foods&nutrition"

print(f"🚀 Starting high-speed upload to namespace: '{namespace_name}'...")

# Loop through the data in chunks of 100 rows
for i in range(0, len(df), batch_size):
    # Slice a batch of 100 rows
    batch_df = df.iloc[i : i + batch_size]

    texts = batch_df['text_chunk'].astype(str).tolist()
    vectors = model.encode(texts, batch_size=batch_size, show_progress_bar=False).tolist()

    records_to_upload = []

    batch_data = batch_df.to_dict('records')

    for j, row in enumerate(batch_data):
        record_id = str(row['fdc_id'])

        metadata = {
            'search_query': str(row['search_query']),
            'food_name': str(row['food_name']),
            'food_category': str(row['food_category']),
            'data_type': str(row['data_type']),
            'text_chunk': str(row['text_chunk']),

            # Numeric values
            'calories': float(row['calories']),
            'protein_g': float(row['protein_g']),
            'carbs_g': float(row['carbs_g']),
            'fat_g': float(row['fat_g']),
            'fiber_g': float(row['fiber_g']),
            'sugar_g': float(row['sugar_g']),
            'saturated_fat_g': float(row['saturated_fat_g']),
            'cholesterol_mg': float(row['cholesterol_mg']),
            'sodium_mg': float(row['sodium_mg']),
            'iron_mg': float(row['iron_mg']),
            'calcium_mg': float(row['calcium_mg']),
            'potassium_mg': float(row['potassium_mg']),
            'magnesium_mg': float(row['magnesium_mg']),
            'vitamin_c_mg': float(row['vitamin_c_mg']),
            'vitamin_b12_ug': float(row['vitamin_b12_ug'])
        }

        records_to_upload.append((record_id, vectors[j], metadata))

    index.upsert(vectors=records_to_upload, namespace=namespace_name)

    if (i + batch_size) % 500 == 0 or (i + batch_size) >= len(df):
        print(f"Uploaded {min(i + batch_size, len(df))} / {len(df)} rows...")

print("FINAL BATCH UPLOADED SUCCESSFULLY!")

Loading model on: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

🚀 Starting high-speed upload to namespace: 'foods&nutrition'...
Uploaded 76 / 76 rows...
FINAL BATCH UPLOADED SUCCESSFULLY!


## GYM and Exercise

In [ ]:
import os
import shutil
file_path="/content/drive/MyDrive/SPORT_METADATA/Structure data/Gym_exercises/exercise_merged_normalized_data.csv"
dist_path="/content/Gym&exercise_data/"

os.makedirs(dist_path,exist_ok=True)

shutil.copy2(file_path,dist_path)

'/content/Gym&exercise_data/exercise_merged_normalized_data.csv'

In [ ]:
df_gym_exercise=pd.read_csv("/content/Gym&exercise_data/exercise_merged_normalized_data.csv")

print("the columnsof the exercise data\n",df_gym_exercise.columns.tolist())

the columnsof the exercise data
 ['Unnamed: 0', 'exercise_name', 'category', 'primary_muscles', 'secondary_muscles', 'equipment', 'mechanic', 'force', 'level', 'source_url', 'source', 'text_chunk']


In [ ]:

## check for the columns
print(df_gym_exercise.loc[1000,"Unnamed: 0"])

## that is index column drop it
df_gym_exercise.drop(columns="Unnamed: 0",inplace=True)

1000


In [ ]:
df_gym_exercise.isna().sum()

exercise_name        0
category             0
primary_muscles      0
secondary_muscles    0
equipment            0
mechanic             0
force                0
level                0
source_url           0
source               0
text_chunk           0
dtype: int64

In [ ]:
df_gym_exercise.iloc[0]

exercise_name                                          180° jump turns
category                                                   Plyometrics
primary_muscles                                          Not Specified
secondary_muscles                                        Not Specified
equipment                                                Not Specified
mechanic                                                      Compound
force                                                             Push
level                                                    Not Specified
source_url                   https://exrx.net/Plyometrics/180JumpTurns
source                                                    exrx_scraper
text_chunk           Utility: Plyometric. Comments: Maintain athlet...
Name: 0, dtype: str

In [ ]:
import pandas as pd
from pinecone import Pinecone
from sentence_transformers import SentenceTransformer
import torch

PINECONE_API_KEY = "######"
pc = Pinecone(api_key=PINECONE_API_KEY)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Loading model on: {device}")

model = SentenceTransformer('BAAI/bge-large-en-v1.5', device=device)

index_name = "medical-sport-assistant"
index = pc.Index(index_name)

df = pd.read_csv("/content/Gym&exercise_data/exercise_merged_normalized_data.csv")

# 4. Fast Batch Processing & Upload
batch_size = 100
namespace_name = "gym&exercises"

print(f"🚀 Starting high-speed upload for {len(df)} exercises to namespace: '{namespace_name}'...")

for i in range(0, len(df), batch_size):
    batch_df = df.iloc[i : i + batch_size]

    texts = batch_df['text_chunk'].astype(str).tolist()
    vectors = model.encode(texts, batch_size=batch_size, show_progress_bar=False).tolist()

    records_to_upload = []

    batch_data = batch_df.to_dict('records')

    for j, row in enumerate(batch_data):
        record_id = f"ex_{i+j}"

        metadata = {
            'exercise_name': str(row['exercise_name']),
            'category': str(row['category']),
            'primary_muscles': str(row['primary_muscles']),
            'secondary_muscles': str(row['secondary_muscles']),
            'equipment': str(row['equipment']),
            'mechanic': str(row['mechanic']),
            'force': str(row['force']),
            'level': str(row['level']),
            'source_url': str(row['source_url']),
            'source': str(row['source']),
            'text_chunk': str(row['text_chunk'])
        }

        records_to_upload.append((record_id, vectors[j], metadata))

    index.upsert(vectors=records_to_upload, namespace=namespace_name)

    if (i + batch_size) % 500 == 0 or (i + batch_size) >= len(df):
        print(f"✅ Uploaded {min(i + batch_size, len(df))} / {len(df)} exercises...")

print(f"✨ FINAL BATCH UPLOADED SUCCESSFULLY TO '{namespace_name}'!")

Loading model on: cuda


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🚀 Starting high-speed upload for 1506 exercises to namespace: 'gym&exercises'...
✅ Uploaded 500 / 1506 exercises...
✅ Uploaded 1000 / 1506 exercises...
✅ Uploaded 1500 / 1506 exercises...
✅ Uploaded 1506 / 1506 exercises...
✨ FINAL BATCH UPLOADED SUCCESSFULLY TO 'gym&exercises'!


## warmup ground


In [ ]:

file_path="/content/drive/MyDrive/SPORT_METADATA/Structure data/warm_up/playground/fifa11plus_enriched_rag_ready.csv"
dist_path="/content/warm_up/playground/"

os.makedirs(dist_path,exist_ok=True)

shutil.copy2(file_path,dist_path)


'/content/warm_up/playground/fifa11plus_enriched_rag_ready.csv'

In [ ]:
df_warmyp=pd.read_csv("/content/warm_up/playground/fifa11plus_enriched_rag_ready.csv")

df_warmyp.isna().sum()

exercise_name       0
phase               0
duration_or_reps    0
instructions        0
source              0
primary_muscles     0
equipment           6
primary_action      0
movement_type       0
domain              0
dtype: int64

In [ ]:
df_warmyp["equipment"]=df_warmyp["equipment"].fillna("not specified")
df_warmyp.isna().sum()

exercise_name       0
phase               0
duration_or_reps    0
instructions        0
source              0
primary_muscles     0
equipment           0
primary_action      0
movement_type       0
domain              0
dtype: int64

In [ ]:
df_warmyp['text_chunk'] = df_warmyp['exercise_name'] + " (" + df_warmyp['phase'] + "). " + df_warmyp['instructions']
df_warmyp.to_csv("/content/warm_up/playground/football_warmups_text_chunk.csv",index=False)

In [ ]:
df_warmyp.loc[0,"text_chunk"]

'RUNNING STRAIGHT AHEAD (Raise / Activate (Part 1)). Important when performing the exercise: Jog straight to the last cone. Run slightly more quickly on the way back. Do the exercise twice. 1 Make sure you keep your upper body straight. 2 Your hips, knees and feet should be aligned. Do not let your knees buckle inwards.'

In [ ]:
df_warmyp.iloc[0]

exercise_name                                  RUNNING STRAIGHT AHEAD
phase                                       Raise / Activate (Part 1)
duration_or_reps                                                  1 M
instructions        Important when performing the exercise: Jog st...
source                                                FIFA 11+ Manual
primary_muscles                                       Full Lower Body
equipment                                                       Cones
primary_action                                       general_movement
movement_type                                                 Aerobic
domain                                               Football / Pitch
text_chunk          RUNNING STRAIGHT AHEAD (Raise / Activate (Part...
Name: 0, dtype: str

In [ ]:
import pandas as pd
from pinecone import Pinecone
from sentence_transformers import SentenceTransformer
import torch

PINECONE_API_KEY = "######"
pc = Pinecone(api_key=PINECONE_API_KEY)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Loading model on: {device}")
model = SentenceTransformer('BAAI/bge-large-en-v1.5', device=device)

index_name = "medical-sport-assistant"
index = pc.Index(index_name)

df = pd.read_csv("/content/warm_up/playground/football_warmups_text_chunk.csv")
df = df.fillna("Not Specified")

batch_size = 100
namespace_name = "football_warmups"

print(f"Starting high-speed upload for {len(df)} warm-ups to namespace: '{namespace_name}'...")

for i in range(0, len(df), batch_size):
    # Slice a batch of 100 rows
    batch_df = df.iloc[i : i + batch_size]

    # ⚡ GPU BATCHING: Encode all 100 texts at the exact same time
    texts = batch_df['text_chunk'].astype(str).tolist()
    vectors = model.encode(texts, batch_size=batch_size, show_progress_bar=False).tolist()

    records_to_upload = []

    # Convert batch to dictionary format
    batch_data = batch_df.to_dict('records')

    for j, row in enumerate(batch_data):
        # Generate a unique ID (fw = football warmup)
        record_id = f"fw_{i+j}"

        metadata = {
            'exercise_name': str(row['exercise_name']),
            'phase': str(row['phase']),
            'duration_or_reps': str(row['duration_or_reps']),
            'instructions': str(row['instructions']),
            'source': str(row['source']),
            'primary_muscles': str(row['primary_muscles']),
            'equipment': str(row['equipment']),
            'primary_action': str(row['primary_action']),
            'movement_type': str(row['movement_type']),
            'domain': str(row['domain']),
            'text_chunk': str(row.get('text_chunk', ''))
        }

        records_to_upload.append((record_id, vectors[j], metadata))

    index.upsert(vectors=records_to_upload, namespace=namespace_name)

    if (i + batch_size) % 500 == 0 or (i + batch_size) >= len(df):
        print(f"Uploaded {min(i + batch_size, len(df))} / {len(df)} warm-ups...")

print(f"FINAL BATCH UPLOADED SUCCESSFULLY TO '{namespace_name}'!")

Loading model on: cuda


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Starting high-speed upload for 12 warm-ups to namespace: 'football_warmups'...
Uploaded 12 / 12 warm-ups...
FINAL BATCH UPLOADED SUCCESSFULLY TO 'football_warmups'!


## warmup general exercise

In [ ]:
import shutil
import os
file_path="/content/drive/MyDrive/SPORT_METADATA/Structure data/warm_up/Gym/clean/warmup_gym.csv"
dist_path="/content/warm_up/gym/"

os.makedirs(dist_path,exist_ok=True)

shutil.copy2(file_path,dist_path)

'/content/warm_up/gym/warmup_gym.csv'

In [ ]:
import pandas as pd
df_warmup_gym=pd.read_csv("/content/warm_up/gym/warmup_gym.csv",encoding='windows-1252')
df_warmup_gym.isna().sum()

Unnamed: 0              0
source_url              0
exercise_name           0
classification          9
bearing                 3
impact                  3
instructions            9
preparation             0
execution               0
comments                0
force_(articulation)    9
dynamic                 1
utility                 6
mechanics               6
force                   6
easier                  8
harder                  8
static                  3
muscles                 7
dtype: int64

In [ ]:
df_warmup_gym.drop(columns=["Unnamed: 0","force_(articulation)"],inplace=True)
df_warmup_gym=df_warmup_gym.fillna("not specified")
df_warmup_gym.isna().sum()

source_url        0
exercise_name     0
classification    0
bearing           0
impact            0
instructions      0
preparation       0
execution         0
comments          0
dynamic           0
utility           0
mechanics         0
force             0
easier            0
harder            0
static            0
muscles           0
dtype: int64

In [ ]:
print("columns for data warmup gym\n",df_warmup_gym.columns.tolist())

columns for data warmup gym
 ['source_url', 'exercise_name', 'classification', 'bearing', 'impact', 'instructions', 'preparation', 'execution', 'comments', 'dynamic', 'utility', 'mechanics', 'force', 'easier', 'harder', 'static', 'muscles']


In [ ]:
df_warmup_gym['text_chunk'] = df_warmup_gym['exercise_name'] + ". Preparation: " + df_warmup_gym['preparation'] + " Execution: " + df_warmup_gym['execution'] + " Comments: " + df_warmup_gym['comments']

In [ ]:
df_warmup_gym.loc[0,"text_chunk"]

'Jumping Jack. Preparation: Stand with feet together, knees slightly bent, and arms to sides. Execution: Jump while raising arms and separating legs to sides. Land on forefoot with legs apart and arms overhead. Jump again while lower arms and returning legs to midline. Land on forefoot with arms and legs in original position and repeat. Comments: Intensity can be increased by jumping faster.'

In [ ]:
df_warmup_gym.to_csv("/content/warm_up/gym/warmupgym.csv")

In [ ]:
df_warmup_gym.iloc[0]

source_url           https://exrx.net/Aerobic/Exercises/JumpingJack
exercise_name                                          Jumping Jack
classification                                        not specified
bearing                                                      Weight
impact                                                         High
instructions                                          not specified
preparation       Stand with feet together, knees slightly bent,...
execution         Jump while raising arms and separating legs to...
comments              Intensity can be increased by jumping faster.
dynamic           Ankle:\n                                      ...
utility                                               not specified
mechanics                                             not specified
force                                                 not specified
easier                                                not specified
harder                                          

In [ ]:
import pandas as pd
from pinecone import Pinecone
from sentence_transformers import SentenceTransformer
import torch

PINECONE_API_KEY = "######"
pc = Pinecone(api_key=PINECONE_API_KEY)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Loading model on: {device}")
model = SentenceTransformer('BAAI/bge-large-en-v1.5', device=device)

index_name = "medical-sport-assistant"
index = pc.Index(index_name)

df = pd.read_csv("/content/warm_up/gym/warmupgym.csv")

batch_size = 100
namespace_name = "gym_warmups"

print(f"🚀 Starting high-speed upload for {len(df)} warm-ups to namespace: '{namespace_name}'...")

for i in range(0, len(df), batch_size):
    # Slice a batch of 100 rows
    batch_df = df.iloc[i : i + batch_size]

    texts = batch_df['text_chunk'].astype(str).tolist()
    vectors = model.encode(texts, batch_size=batch_size, show_progress_bar=False).tolist()

    records_to_upload = []

    batch_data = batch_df.to_dict('records')

    for j, row in enumerate(batch_data):
        # Generate a unique ID (gw = gym warmup)
        record_id = f"gw_{i+j}"

        metadata = {
            'source_url': str(row['source_url']),
            'exercise_name': str(row['exercise_name']),
            'classification': str(row['classification']),
            'bearing': str(row['bearing']),
            'impact': str(row['impact']),
            'instructions': str(row['instructions']),
            'preparation': str(row['preparation']),
            'execution': str(row['execution']),
            'comments': str(row['comments']),
            'dynamic': str(row['dynamic']),
            'utility': str(row['utility']),
            'mechanics': str(row['mechanics']),
            'force': str(row['force']),
            'easier': str(row['easier']),
            'harder': str(row['harder']),
            'static': str(row['static']),
            'muscles': str(row['muscles']),
            'text_chunk': str(row.get('text_chunk', ''))
        }

        records_to_upload.append((record_id, vectors[j], metadata))

    index.upsert(vectors=records_to_upload, namespace=namespace_name)

    if (i + batch_size) % 500 == 0 or (i + batch_size) >= len(df):
        print(f"Uploaded {min(i + batch_size, len(df))} / {len(df)} gym warm-ups...")

print(f"FINAL BATCH UPLOADED SUCCESSFULLY TO '{namespace_name}'!")

Loading model on: cuda


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🚀 Starting high-speed upload for 9 warm-ups to namespace: 'gym_warmups'...
Uploaded 9 / 9 gym warm-ups...
FINAL BATCH UPLOADED SUCCESSFULLY TO 'gym_warmups'!


## rehab and recovery

In [ ]:
file_path="/content/drive/MyDrive/SPORT_METADATA/Structure data/Rehab&Recovery/Clean Rehab /4_5879831926398786944.json"
dist_path="/content/rehab_recovery/"

os.makedirs(dist_path,exist_ok=True)
shutil.copy2(file_path,dist_path)

'/content/rehab_recovery/4_5879831926398786944.json'

In [ ]:
import json
import re
from rich.console import Console
from rich.tree import Tree
from rich.text import Text
from rich.panel import Panel
from rich.style import Style

console = Console()

def natural_sort_key(item):
    section_name = item[0]
    # Look for a number at the start of the section name
    match = re.search(r'^(\d+)', section_name)
    if match:
        return int(match.group(1))
    return 999  # If no number, push to the bottom

# Load your generated JSON file
try:
    with open('/content/rehab_recovery/4_5879831926398786944.json', 'r', encoding='utf-8') as f:
        master_data = json.load(f)
except FileNotFoundError:
    console.print("[bold red]Error: ' not found.[/bold red]")
    exit()

# Create the root of the tree - using bright white for better visibility
root = Tree("[black]PHYSIO-PEDIA INJURY DATABASE[/black]", guide_style="white")

# Loop through each parent (injury) and its children (sections)
for injury_index, injury in enumerate(master_data, 1):
    injury_name = injury.get("injury_name", "Unknown Injury")

    # Add the Parent node with numbering - using bright white for maximum visibility
    injury_node = root.add(f"[black]{injury_index}. {injury_name}[/black]")

    # Add the Child nodes (sections)
    sections = injury.get("sections", {})
    if not sections:
        injury_node.add("[dim]No content available[/dim]")
        continue

    # Sort sections using our natural sort key
    sorted_sections = sorted(sections.items(), key=natural_sort_key)

    for section_index, (section_name, content_list) in enumerate(sorted_sections, 1):
        item_count = len(content_list)

        # Create section display with high contrast colors
        if item_count > 0:
            section_text = Text()
            section_text.append(f"{section_index}. {section_name} ", style="cyan")  # Cyan for good visibility
            section_text.append(f"[{item_count}]", style="yellow")  # Yellow for counts
            injury_node.add(section_text)
        else:
            section_text = Text(f"{section_index}. {section_name} ", style="dim white")
            section_text.append("[empty]", style="red")
            injury_node.add(section_text)

# Print the enhanced tree to the console
console.print("\n")
console.print(root)
console.print("\n")

# Add summary statistics at the bottom
total_injuries = len(master_data)
total_sections = sum(len(injury.get("sections", {})) for injury in master_data)
total_blocks = sum(
    len(content)
    for injury in master_data
    for content in injury.get("sections", {}).values()
)

summary_panel = Panel(
    f"[black]Total Injuries:[/black] {total_injuries}   |   "
    f"[black]Total Sections:[/black] {total_sections}   |   "
    f"[black]Total Content Blocks:[/black] {total_blocks}",
    title="Database Summary",
    border_style="white",
    expand=False
)
console.print(summary_panel)

PHYSIO-PEDIA INJURY DATABASE
├── 1. Hamstring Strain
│   ├── 1. 1_Introduction_Definition [6]
│   ├── 2. 3_Mechanism_Etiology [2]
│   ├── 3. 4_Risk_Factors [6]
│   ├── 4. 5_Clinical_Presentation [6]
│   ├── 5. 6_Examination_Diagnosis [23]
│   ├── 6. 7_Differential_Diagnosis [6]
│   ├── 7. 8_Medical_Management [13]
│   ├── 8. 9_Rehabilitation_Physiotherapy [32]
│   └── 9. 10_Prevention_Return_To_Play [8]
├── 2. Groin Strain
│   ├── 1. 1_Introduction_Definition [5]
│   ├── 2. 2_Anatomy [17]
│   ├── 3. 3_Mechanism_Etiology [15]
│   ├── 4. 5_Clinical_Presentation [6]
│   ├── 5. 6_Examination_Diagnosis [7]
│   ├── 6. 7_Differential_Diagnosis [6]
│   ├── 7. 8_Medical_Management [8]
│   ├── 8. 9_Rehabilitation_Physiotherapy [2]
│   └── 9. 10_Prevention_Return_To_Play [9]
├── 3. Calf Strain
│   ├── 1. 1_Introduction_Definition [3]
│   ├── 2. 2_Anatomy [4]
│   ├── 3. 3_Mechanism_Etiology [18]
│   ├── 4. 4_Risk_Factors [6]
│   ├── 5. 5_Clinical_Presentation [4]
│   ├── 6. 6_Examination_Diagnosis [2]
│   ├── 7. 7_Differential_Diagnosis [3]
│   ├── 8. 8_Medical_Management [5]
│   ├── 9. 9_Rehabilitation_Physiotherapy [12]
│   └── 10. 10_Prevention_Return_To_Play [3]
├── 4. Hip Flexor Strain
│   ├── 1. 1_Introduction_Definition [3]
│   ├── 2. 2_Anatomy [15]
│   ├── 3. 3_Mechanism_Etiology [11]
│   ├── 4. 5_Clinical_Presentation [2]
│   ├── 5. 6_Examination_Diagnosis [27]
│   ├── 6. 7_Differential_Diagnosis [4]
│   ├── 7. 8_Medical_Management [8]
│   ├── 8. 9_Rehabilitation_Physiotherapy [6]
│   └── 9. 10_Prevention_Return_To_Play [4]
├── 5. Quadriceps Contusion
│   ├── 1. 1_Introduction_Definition [3]
│   ├── 2. 3_Mechanism_Etiology [10]
│   ├── 3. 4_Risk_Factors [2]
│   ├── 4. 5_Clinical_Presentation [4]
│   ├── 5. 6_Examination_Diagnosis [6]
│   ├── 6. 9_Rehabilitation_Physiotherapy [6]
│   └── 7. 10_Prevention_Return_To_Play [4]
├── 6. Metatarsal Fracture
│   ├── 1. 1_Introduction_Definition [2]
│   ├── 2. 2_Anatomy [2]
│   ├── 3. 3_Mechanism_Etiology [8]
│   ├── 4. 5_Clinical_Presentation [4]
│   ├── 5. 6_Examination_Diagnosis [16]
│   ├── 6. 7_Differential_Diagnosis [6]
│   ├── 7. 8_Medical_Management [2]
│   └── 8. 9_Rehabilitation_Physiotherapy [6]
├── 7. Concussion
│   ├── 1. 1_Introduction_Definition [4]
│   ├── 2. 3_Mechanism_Etiology [2]
│   ├── 3. 4_Risk_Factors [6]
│   ├── 4. 5_Clinical_Presentation [12]
│   ├── 5. 6_Examination_Diagnosis [55]
│   ├── 6. 8_Medical_Management [2]
│   ├── 7. 9_Rehabilitation_Physiotherapy [20]
│   └── 8. 10_Prevention_Return_To_Play [6]
├── 8. ACL Injury
│   ├── 1. 1_Introduction_Definition [2]
│   ├── 2. 2_Anatomy [9]
│   ├── 3. 3_Mechanism_Etiology [70]
│   ├── 4. 4_Risk_Factors [20]
│   ├── 5. 5_Clinical_Presentation [2]
│   ├── 6. 6_Examination_Diagnosis [35]
│   ├── 7. 7_Differential_Diagnosis [7]
│   ├── 8. 9_Rehabilitation_Physiotherapy [18]
│   └── 9. 10_Prevention_Return_To_Play [10]
├── 9. Meniscus Lesion
│   ├── 1. 1_Introduction_Definition [2]
│   ├── 2. 2_Anatomy [5]
│   ├── 3. 3_Mechanism_Etiology [22]
│   ├── 4. 4_Risk_Factors [2]
│   ├── 5. 5_Clinical_Presentation [3]
│   ├── 6. 6_Examination_Diagnosis [25]
│   ├── 7. 7_Differential_Diagnosis [6]
│   ├── 8. 8_Medical_Management [21]
│   ├── 9. 9_Rehabilitation_Physiotherapy [25]
│   └── 10. 10_Prevention_Return_To_Play [4]
├── 10. MCL Sprain
│   ├── 1. 1_Introduction_Definition [2]
│   ├── 2. 3_Mechanism_Etiology [2]
│   ├── 3. 5_Clinical_Presentation [6]
│   ├── 4. 6_Examination_Diagnosis [14]
│   ├── 5. 7_Differential_Diagnosis [6]
│   ├── 6. 8_Medical_Management [11]
│   ├── 7. 9_Rehabilitation_Physiotherapy [18]
│   └── 8. 10_Prevention_Return_To_Play [4]
├── 11. Patellar Tendinopathy
│   ├── 1. 1_Introduction_Definition [3]
│   ├── 2. 2_Anatomy [6]
│   ├── 3. 3_Mechanism_Etiology [8]
│   ├── 4. 5_Clinical_Presentation [5]
│   ├── 5. 6_Examination_Diagnosis [7]
│   ├── 6. 7_Differential_Diagnosis [2]
│   ├── 7. 8_Medical_Management [11]
│   ├── 8. 9_Rehabilitation_Physiotherapy [13]
│   └── 9. 10_Prevention_Return_To_Play [2]

╭────────────────────────────── Database Summary ───────────────────────────────╮
│ Total Injuries: 16   |   Total Sections: 137   |   Total Content Blocks: 1280 │
╰───────────────────────────────────────────────────────────────────────────────╯

In [ ]:
import json
import torch
from pinecone import Pinecone
from sentence_transformers import SentenceTransformer

# 1. Setup Pinecone API
PINECONE_API_KEY = "######"
pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index("medical-sport-assistant")

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Loading model on: {device}")
model = SentenceTransformer('BAAI/bge-large-en-v1.5', device=device)



# 3. Load your standardized database
json_file_path = "/content/rehab_recovery/4_5879831926398786944.json"
with open(json_file_path, "r", encoding='utf-8') as file:
    rehab_data = json.load(file)

flat_records = []

# 🛠️ THE ULTIMATE SPONGE EXTRACTOR
# No length limits. No dropped headers. It grabs EVERYTHING.
def extract_absolutely_everything(data):
    extracted = []
    if isinstance(data, dict):
        if "text" in data and isinstance(data["text"], str):
            extracted.append(data["text"].strip())
        for key, value in data.items():
            if key != "text":
                extracted.extend(extract_absolutely_everything(value))
    elif isinstance(data, list):
        for item in data:
            extracted.extend(extract_absolutely_everything(item))
    elif isinstance(data, str) and data.strip():
        # Grab EVERY string that isn't just blank space!
        extracted.append(data.strip())

    return extracted

# 4. Extract data and format it beautifully
for injury in rehab_data:
    injury_name = injury.get("injury_name", "Unknown Injury")
    sections = injury.get("sections", {})

    for section_name, blocks in sections.items():
        if isinstance(blocks, list):
            for block in blocks:
                # Grab literally every piece of text in this block
                raw_texts = extract_absolutely_everything(block)

                formatted_texts = []
                for t in raw_texts:
                    if t.startswith("==="):
                        # Clean up the header so it looks like a nice title for the AI
                        clean_title = t.replace("===", "").strip()
                        formatted_texts.append(f"\n[{clean_title.upper()}]")
                    else:
                        # Add a clean bullet point to everything else (paragraphs, symptoms, exercises)
                        formatted_texts.append(f"• {t}")

                if formatted_texts:
                    # Glue it all together into one massive, context-rich chunk
                    combined_chunk = "\n".join(formatted_texts).strip()

                    flat_records.append({
                        "injury_name": str(injury_name),
                        "section_name": str(section_name),
                        "text_chunk": combined_chunk
                    })

print(f"✅ Successfully extracted {len(flat_records)} comprehensive medical blocks! ZERO data dropped.")

# 5. Fast Batch Processing & Upload
batch_size = 100
namespace_name = "physio_rehab"

print(f"🚀 Starting high-speed upload to namespace: '{namespace_name}'...")

for i in range(0, len(flat_records), batch_size):
    batch_data = flat_records[i : i + batch_size]

    texts = [record['text_chunk'] for record in batch_data]
    vectors = model.encode(texts, batch_size=batch_size, show_progress_bar=False).tolist()

    records_to_upload = []

    for j, record in enumerate(batch_data):
        record_id = f"ph_{i+j}"

        metadata = {
            'injury_name': record['injury_name'],
            'section_name': record['section_name'],
            'text_chunk': record['text_chunk']
        }

        records_to_upload.append((record_id, vectors[j], metadata))

    index.upsert(vectors=records_to_upload, namespace=namespace_name)

    if (i + batch_size) % 500 == 0 or (i + batch_size) >= len(flat_records):
        print(f"✅ Uploaded {min(i + batch_size, len(flat_records))} / {len(flat_records)} rehab chunks...")

print(f"✨ FINAL BATCH UPLOADED SUCCESSFULLY TO '{namespace_name}'!")

Loading model on: cuda


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Successfully extracted 1280 comprehensive medical blocks! ZERO data dropped.
🚀 Starting high-speed upload to namespace: 'physio_rehab'...
✅ Uploaded 500 / 1280 rehab chunks...
✅ Uploaded 1000 / 1280 rehab chunks...
✅ Uploaded 1280 / 1280 rehab chunks...
✨ FINAL BATCH UPLOADED SUCCESSFULLY TO 'physio_rehab'!


In [ ]:
import torch
from pinecone import Pinecone
from sentence_transformers import SentenceTransformer

# 1. Connect to Pinecone
PINECONE_API_KEY = "######"
pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index("medical-sport-assistant")

# 2. Load the BAAI Model
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = SentenceTransformer('BAAI/bge-large-en-v1.5', device=device)

# 3. Create a query designed to pull out a Rehab timeline
print("🔍 Fetching a recovery protocol block from Pinecone...\n")
query_vector = model.encode("Phase 1 rehabilitation timeline and exercises").tolist()

# 4. Ask Pinecone for just ONE chunk
response = index.query(
    namespace="physio_rehab",
    vector=query_vector,
    top_k=1,
    include_metadata=True
)

# 5. Display the metadata exactly as it is stored
if response['matches']:
    match = response['matches'][0]
    meta = match['metadata']

    print("✅ HERE IS ONE COMPLETE ROW OF METADATA:")
    print("=" * 60)
    print(f"📌 Record ID:   {match['id']}")
    print(f"🩺 Injury Name: {meta.get('injury_name')}")
    print(f"📂 Section:     {meta.get('section_name')}")
    print("-" * 60)
    print("📝 EXACT TEXT CHUNK STORED FOR THE AI TO READ:")
    print(meta.get('text_chunk'))
    print("=" * 60)
else:
    print("❌ No data found.")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🔍 Fetching a recovery protocol block from Pinecone...

✅ HERE IS ONE COMPLETE ROW OF METADATA:
📌 Record ID:   ph_174
🩺 Injury Name: Groin Strain
📂 Section:     8_Medical_Management
------------------------------------------------------------
📝 EXACT TEXT CHUNK STORED FOR THE AI TO READ:
• The primary goal of the treatment program is to minimize the effects of immobilization, regain full range of motion, and restore full muscle strength, endurance and coordination. Therefore, crutches, local cold application, and anti-inflammatory medication are recommended in the initial phase. Muscle exercises can usually be started early, but training should be performed within the limits of pain with careful isometric contractions against resistance.
• After the initial phase, heat is usually valuable, especially when muscle training is started. In general, exercises are performed in a pain-free range of motion and increased pain should not occur after activity.
• As rehabilitation progresses, mild 

In [ ]:
import torch
from pinecone import Pinecone
from sentence_transformers import SentenceTransformer

# 1. Connect to Pinecone
PINECONE_API_KEY = "######"
pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index("medical-sport-assistant")

# 2. Load the BAAI Model
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = SentenceTransformer('BAAI/bge-large-en-v1.5', device=device)

# 3. Create a simple test question
question = "What is a Hamstring Strain?"
print(f"🔍 Searching for: '{question}'...\n")

# Convert the question to a vector
query_vector = model.encode(question).tolist()

# 4. Ask Pinecone for the single best match (top_k=1)
response = index.query(
    namespace="physio_rehab",
    vector=query_vector,
    top_k=1,
    include_metadata=True
)

# 5. Display the result clearly
if response['matches']:
    match = response['matches'][0] # Grab the very first result

    print("✅ HERE IS YOUR DATA INSIDE PINECONE:")
    print("-" * 50)
    print(f"📌 Record ID:   {match['id']}")
    print(f"🎯 Score:       {match['score']:.4f} (Cosine Similarity)")
    print(f"🩺 Injury Name: {match['metadata']['injury_name']}")
    print(f"📂 Section:     {match['metadata']['section_name']}")
    print(f"📝 Text Chunk:  {match['metadata']['text_chunk']}")
    print("-" * 50)
else:
    print("❌ No data found. Check your namespace name!")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🔍 Searching for: 'What is a Hamstring Strain?'...

✅ HERE IS YOUR DATA INSIDE PINECONE:
--------------------------------------------------
📌 Record ID:   ph_6
🎯 Score:       0.8118 (Cosine Similarity)
🩺 Injury Name: Hamstring Strain
📂 Section:     1_Introduction_Definition
📝 Text Chunk:  Hamstring strains are caused by a rapid extensive contraction or a violent stretch of the hamstring muscle group which causes high mechanical stress. This type of injury presents as sudden pain in the back of the thigh with popping or pulling sensation due to hamstring muscle fiber disruption, without direct external contact to the thigh. Hamstring strains are the most commonly reported muscle injury in sports and biceps femoris is highly injured recurrently or for the first time. The athletes with the history of hamstring stain injury have 3.6% higher chances to get the recurrence.
--------------------------------------------------


In [ ]:
import torch
from pinecone import Pinecone
from sentence_transformers import SentenceTransformer

# 1. Connect to Pinecone
PINECONE_API_KEY = "######"
pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index("medical-sport-assistant")

# 2. Load the BAAI Model
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = SentenceTransformer('BAAI/bge-large-en-v1.5', device=device)

# 3. Define the exact injury we want to investigate
target_injury = "Hamstring Strain"
print(f"🔍 Fetching all sections for: '{target_injury}'...\n")

# We just encode the name of the injury to get a baseline vector
query_vector = model.encode(target_injury).tolist()

# 4. Ask Pinecone for 50 chunks, but FILTER for only this injury!
response = index.query(
    namespace="physio_rehab",
    vector=query_vector,
    top_k=50,
    include_metadata=True,
    filter={
        "injury_name": {"$eq": target_injury} # This is the magic filter!
    }
)

# 5. Group the results by Section Name
sections_found = {}

for match in response['matches']:
    section = match['metadata']['section_name']
    text = match['metadata']['text_chunk']

    # If we haven't seen this section yet, add it to our dictionary
    if section not in sections_found:
        sections_found[section] = []

    sections_found[section].append(text)

# 6. Display the organized data beautifully
if sections_found:
    print(f"✅ FOUND {len(sections_found)} DIFFERENT SECTORS FOR {target_injury.upper()}:")
    print("=" * 60)

    # Sort them alphabetically so they print in order (1_Intro, 2_Anatomy, etc.)
    for section in sorted(sections_found.keys()):
        chunk_count = len(sections_found[section])

        # Grab just the first 100 characters of the first chunk to show a preview
        preview_text = sections_found[section][0][:100] + "..."

        print(f"📂 {section} (Contains {chunk_count} text chunks)")
        print(f"   ↳ Preview: {preview_text}\n")
    print("=" * 60)
else:
    print(f"No data found for {target_injury}.")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🔍 Fetching all sections for: 'Hamstring Strain'...

✅ FOUND 9 DIFFERENT SECTORS FOR HAMSTRING STRAIN:
📂 10_Prevention_Return_To_Play (Contains 3 text chunks)
   ↳ Preview: Injury prevention exercise program should include Nordic hamstring exercise with other components li...

📂 1_Introduction_Definition (Contains 4 text chunks)
   ↳ Preview: Hamstring strains are common in sports with a dynamic character like sprinting, jumping, and contact...

📂 3_Mechanism_Etiology (Contains 1 text chunks)
   ↳ Preview: The cause of a hamstring muscle strain is often obscure. In the second half of the swing phase, the ...

📂 4_Risk_Factors (Contains 5 text chunks)
   ↳ Preview: During activities like running and kicking, hamstring will lengthen with concurrent hip flexion and ...

📂 5_Clinical_Presentation (Contains 4 text chunks)
   ↳ Preview: Decreased strength on isometric contraction
- Decreased length of the hamstrings...

📂 6_Examination_Diagnosis (Contains 11 text chunks)
   ↳ Preview: Running

## book chunk meta data

In [ ]:
import os
import shutil

file_path="/content/drive/MyDrive/SPORT_METADATA/unstructured data/sport_book/pickle/chunks_backup_20260219.pkl"

dist_path="/content/book_chunk/"

os.makedirs(dist_path,exist_ok=True)

shutil.copy2(file_path,dist_path)

'/content/book_chunk/chunks_backup_20260219.pkl'

In [ ]:
import pickle
import torch
from pinecone import Pinecone
from sentence_transformers import SentenceTransformer

PINECONE_API_KEY = "######"
pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index("medical-sport-assistant")

namespace_name = "sports_library"

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Loading model on: {device}...")
model = SentenceTransformer('BAAI/bge-large-en-v1.5', device=device)

pickle_file_path = "/content/book_chunk/chunks_backup_20260219.pkl"

try:
    with open(pickle_file_path, "rb") as f:
        all_premium_chunks = pickle.load(f)
    print(f"Loaded {len(all_premium_chunks)} chunks from Pickle file.")
except FileNotFoundError:
    print("Could not find the pickle file. Check the path!")
    all_premium_chunks = []


flat_records = []

print("🔄 Standardizing LangChain Documents...")
for chunk in all_premium_chunks:
    # 1. Extract the text
    text = chunk.page_content.replace('\n', ' ').strip()

    # 2. Extract the LangChain metadata
    old_meta = chunk.metadata

    if text:
        # 3. Build our perfectly clean Standard Metadata dictionary
        # Pinecone requires metadata values to be strings, numbers, or lists of strings.
        standard_meta = {
            "source_book": str(old_meta.get("source", "Unknown Book")),
            "category": str(old_meta.get("category", "General")),
            "text_chunk": text
        }

        # Dynamically grab headers, but IGNORE the old "text" key to prevent duplicates
        for key, value in old_meta.items():
            if key not in ["source", "category", "text"] and value is not None:
                standard_meta[str(key)] = str(value)

        flat_records.append(standard_meta)

print(f"✅ Successfully flattened {len(flat_records)} book chunks!")


batch_size = 100

print(f"🚀 Starting high-speed upload to namespace: '{namespace_name}'...")

for i in range(0, len(flat_records), batch_size):
    batch_data = flat_records[i : i + batch_size]

    texts = [record['text_chunk'] for record in batch_data]
    vectors = model.encode(texts, batch_size=batch_size, show_progress_bar=False).tolist()

    records_to_upload = []

    for j, record in enumerate(batch_data):
        record_id = f"book_{i+j}"

        # Attach our standardized metadata
        metadata = record

        records_to_upload.append((record_id, vectors[j], metadata))

    index.upsert(vectors=records_to_upload, namespace=namespace_name)

    if (i + batch_size) % 500 == 0 or (i + batch_size) >= len(flat_records):
        print(f"Uploaded {min(i + batch_size, len(flat_records))} / {len(flat_records)} book chunks...")

print(f"FINAL BATCH UPLOADED SUCCESSFULLY TO '{namespace_name}'!")

Loading model on: cuda...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded 9949 chunks from Pickle file.
🔄 Standardizing LangChain Documents...
✅ Successfully flattened 9949 book chunks!
🚀 Starting high-speed upload to namespace: 'sports_library'...
Uploaded 500 / 9949 book chunks...
Uploaded 1000 / 9949 book chunks...
Uploaded 1500 / 9949 book chunks...
Uploaded 2000 / 9949 book chunks...
Uploaded 2500 / 9949 book chunks...
Uploaded 3000 / 9949 book chunks...
Uploaded 3500 / 9949 book chunks...
Uploaded 4000 / 9949 book chunks...
Uploaded 4500 / 9949 book chunks...
Uploaded 5000 / 9949 book chunks...
Uploaded 5500 / 9949 book chunks...
Uploaded 6000 / 9949 book chunks...
Uploaded 6500 / 9949 book chunks...
Uploaded 7000 / 9949 book chunks...
Uploaded 7500 / 9949 book chunks...
Uploaded 8000 / 9949 book chunks...
Uploaded 8500 / 9949 book chunks...
Uploaded 9000 / 9949 book chunks...
Uploaded 9500 / 9949 book chunks...
